# Housing Price Prediction — Ireland


## Section 1: Environment Setup and Data Loading

This section initializes the computational environment and loads all datasets required for the analysis.

The necessary Python libraries are imported to support data manipulation, visualization, and system operations. NumPy and pandas are used for numerical computations and data handling, while Matplotlib and Seaborn are configured for visualization. Additional libraries such as `os`, `re`, and `time` are included for file handling, regular expressions, and performance tracking.

A fixed random seed is set to ensure reproducibility of results, particularly for any stochastic processes used later in the modelling stage.

The dataset directory is defined using an absolute path constructed via `os.path.expanduser`, ensuring portability across systems. File paths for all datasets — Property Price Register (PPR), Consumer Price Index (CPI), Unemployment, Earnings, and Household Debt — are then created dynamically.

Each dataset is loaded into memory using `pandas.read_csv`. The PPR dataset is read using `latin1` encoding to handle special characters commonly found in address fields.

Finally, an output directory is created within the current working directory to store intermediate results, figures, and model outputs. This ensures that all generated artefacts are organized and easily accessible.

In [26]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import seaborn as sns
import os, re, time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120


# Dataset path
DATA_PATH = os.path.expanduser("~/Desktop/Ireland Price Prediction/Dataset")

PPR_PATH   = os.path.join(DATA_PATH, "Irish Property Price Register (PPR).csv")
CPI_PATH   = os.path.join(DATA_PATH, "Central Statistics Office (CSO) Ireland - CPI.csv")
UNEMP_PATH = os.path.join(DATA_PATH, "Central Statistics Office (CSO) Ireland - Unemployment.csv")
EARN_PATH  = os.path.join(DATA_PATH, "Central Statistics Office (CSO) Ireland - Earnings Hours and Employment Costs Survey.csv")
DEBT_PATH  = os.path.join(DATA_PATH, "Median Debt Service to Income Ratio for Household Main Residence (HMR) DebT.csv")


# Load datasets
ppr_raw   = pd.read_csv(PPR_PATH, encoding="latin1")
cpi_raw   = pd.read_csv(CPI_PATH)
unemp_raw = pd.read_csv(UNEMP_PATH)
earn_raw  = pd.read_csv(EARN_PATH)
debt_raw  = pd.read_csv(DEBT_PATH)

print("All datasets loaded successfully")

# Output directory
WORK_DIR = os.getcwd()
OUT_DIR = os.path.join(WORK_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("Environment ready")

All datasets loaded successfully
Environment ready


## Section 2: Helper Functions and Feature Utilities

This section defines reusable helper functions for data parsing, feature extraction, and model preparation.

### Date Parsing Functions
Custom parsing functions are implemented to convert CSO date formats into standard pandas datetime objects:
- `parse_cso_month`: Converts strings such as "2020 January" into monthly timestamps.
- `parse_quarter`: Converts quarterly representations into the first day of the corresponding quarter.
- `parse_half_year`: Converts half-year indicators into timestamps (January or July).

These functions ensure consistency in time-based data across datasets.

### Address Processing Functions
A set of functions is designed to extract meaningful geographic information from raw address strings:
- `_clean_addr_part`: Removes irrelevant geographic suffixes such as county names or Dublin postal references.
- `_valid_parts`: Splits addresses into components and filters out non-informative elements.
- `extract_suburb`: Identifies the most relevant location unit (typically town or suburb).
- `extract_district`: Extracts a finer-grained locality within the suburb.
- `extract_dublin_pc`: Extracts Dublin postal codes (D01–D24), which provide high spatial precision.
- `extract_eircode_routing`: Extracts the routing key (first three characters) from the Eircode.

These transformations enhance spatial granularity without requiring external geospatial data.

### Bayesian Target Encoding
The `bayesian_oof_encode` function implements an out-of-fold (OOF) Bayesian target encoding approach. This method:
- Computes group-level target means (e.g., average price per suburb)
- Applies smoothing to prevent overfitting for small groups
- Uses cross-validation to eliminate data leakage

This encoding technique is a key driver of predictive performance.

### Evaluation Metrics
The `get_metrics` function computes evaluation metrics including:
- R² (on log-transformed and original scale)
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)

Predictions are transformed back to the original price scale for interpretability.

Overall, this section establishes the foundational tools required for preprocessing, feature engineering, and model evaluation.


In [27]:
print("="*65); print("SECTION 2 — HELPER FUNCTION"); print("="*65)


# ── CSO date parsers ──────────────────────────────────────────────
def parse_cso_month(s):
    try:    return pd.to_datetime(str(s).strip(), format="%Y %B")
    except: return pd.NaT

def parse_quarter(q):
    try:
        y, qt = int(str(q)[:4]), int(str(q)[5])
        return pd.Timestamp(year=y, month=(qt-1)*3+1, day=1)
    except: return pd.NaT

def parse_half_year(s):
    try:
        s = str(s).strip()
        return pd.Timestamp(year=int(s[:4]), month=1 if int(s[5])==1 else 7, day=1)
    except: return pd.NaT

# ── Address feature extraction ────────────────────────────────────
def _clean_addr_part(p):
    """Strip geographic suffix noise from one comma-part of an address."""
    p = p.strip().lower()
    p = re.sub(r'^\s*co\.?\s+\w+\s*$', '', p)          # whole part = 'Co. Dublin'
    p = re.sub(r'\bco\.\s*', '', p)                      # 'co.' within string
    p = re.sub(r'\bcounty\s+\w+\s*$', '', p, flags=re.I) # 'County Dublin' suffix
    p = re.sub(r'\bcounty\s*$', '', p, flags=re.I)
    p = re.sub(r',?\s*dublin\s+\d+\s*$', '', p, flags=re.I)
    p = re.sub(r'\bdublin\s+\d+\s*$', '', p, flags=re.I)
    p = re.sub(r'\s+d\d{1,2}\s*$', '', p, flags=re.I)
    return p.strip()

def _valid_parts(address, county):
    """Return cleaned, meaningful address parts."""
    county_l = str(county).strip().lower()
    if not isinstance(address, str): return []
    raw = [p.strip() for p in address.split(',')]
    result = []
    for p in raw:
        c = _clean_addr_part(p)
        if (len(c) > 2
                and not re.match(r'^\d+[a-z]?\s*$', c.replace(' ', ''))
                and c.lower() not in [county_l, '', 'co', 'co.', 'county']):
            result.append(c)
    return result

def extract_suburb(address, county):
    """Last meaningful part = suburb/town (most reliable location signal)."""
    valid = _valid_parts(address, county)
    return valid[-1] if valid else str(county).strip().lower()

def extract_district(address, county):
    """Second-to-last meaningful part = estate/district within suburb."""
    valid = _valid_parts(address, county)
    return valid[-2] if len(valid) >= 2 else extract_suburb(address, county)

def extract_dublin_pc(address, county):
    """Extract Dublin postcode (D01-D24) — most precise location for Dublin."""
    if str(county).strip().lower() != 'dublin': return 'non_dublin'
    if not isinstance(address, str): return 'dublin_unknown'
    m = re.search(r'\bD(\d{1,2})\b|\bDublin\s+(\d{1,2})\b', address, re.I)
    if m:
        return 'D' + (m.group(1) or m.group(2)).zfill(2)
    return 'dublin_unknown'

def extract_eircode_routing(eircode):
    """First 3 chars of Eircode = routing key (ultra-precise area code)."""
    if pd.isna(eircode) or str(eircode).strip() in ['', 'nan', 'NaN']:
        return 'UNKNOWN'
    return str(eircode).strip().upper()[:3]

# ── Bayesian OOF target encoder ───────────────────────────────────
def bayesian_oof_encode(df, group_col, target_col, global_mean,
                         k=30, f=1.0, n_splits=5):
    """
    Out-of-fold Bayesian smoothed mean encoding — zero data leakage.
    Groups with few samples blend toward global_mean.
    k = inflection point (k samples → 50% weight on group mean).
    """
    from sklearn.model_selection import KFold
    enc = np.full(len(df), global_mean, dtype=np.float64)
    kf  = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    grp = df[group_col].values
    tgt = df[target_col].values
    for tr_idx, val_idx in kf.split(df):
        uniq, inv, counts = np.unique(grp[tr_idx], return_inverse=True, return_counts=True)
        means  = np.bincount(inv, weights=tgt[tr_idx]) / counts
        smooth = 1.0 / (1.0 + np.exp(-(counts - k) / f))
        enc_map = dict(zip(uniq, smooth * means + (1 - smooth) * global_mean))
        enc[val_idx] = np.array([enc_map.get(g, global_mean) for g in grp[val_idx]])
    return enc

# ── Metrics ───────────────────────────────────────────────────────
def get_metrics(ytl, ypl, label=""):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    yt, yp = np.expm1(ytl), np.expm1(ypl)
    return {"Model": label,
            "R² (log)": round(r2_score(ytl, ypl), 4),
            "R² (€)":   round(r2_score(yt, yp),   4),
            "MAE (€)":  round(mean_absolute_error(yt, yp), 0),
            "RMSE (€)": round(np.sqrt(mean_squared_error(yt, yp)), 0)}

print("Helpers defined.")

SECTION 2 — HELPER FUNCTION
Helpers defined.


## Section 3: Data Understanding

This section performs an initial exploration of all datasets to assess their structure, quality, and coverage.

The Property Price Register (PPR) dataset is examined in detail, including:
- Dataset dimensions (number of rows and columns)
- Data types for each variable
- Missing value counts
- Sample records for inspection

This provides an overview of the raw transactional data and highlights potential data quality issues.

The macroeconomic datasets are also briefly explored:
- CPI dataset: Temporal coverage is verified using the first and last recorded months.
- Unemployment dataset: Available metrics are inspected to identify relevant indicators.
- Earnings dataset: Quarterly coverage is confirmed.
- Debt dataset: Overall structure and size are reviewed.

This step ensures that all datasets are correctly loaded and aligned for subsequent preprocessing and integration.


In [28]:
print("="*65); print("SECTION 3 — DATA UNDERSTANDING"); print("="*65)

ppr_raw   = pd.read_csv(PPR_PATH, encoding="latin1")
cpi_raw   = pd.read_csv(CPI_PATH)
unemp_raw = pd.read_csv(UNEMP_PATH)
earn_raw  = pd.read_csv(EARN_PATH)
debt_raw  = pd.read_csv(DEBT_PATH)

print(f"[PPR]  {ppr_raw.shape[0]:,} rows × {ppr_raw.shape[1]} cols")
display(ppr_raw.dtypes.rename("dtype").to_frame())
display(ppr_raw.isnull().sum().rename("missing").to_frame())
display(ppr_raw.head(4))

print(f"\n[CPI] {cpi_raw.shape}  span: {cpi_raw['Month'].iloc[0]} → {cpi_raw['Month'].iloc[-1]}")
print(f"[Unemployment] {unemp_raw.shape}  metrics: {unemp_raw['Statistic Label'].unique().tolist()}")
print(f"[Earnings] {earn_raw.shape}  quarters: {earn_raw['Quarter'].min()} → {earn_raw['Quarter'].max()}")
print(f"[Debt-ratio] {debt_raw.shape}")
print("Data Understanding Completed")


SECTION 3 — DATA UNDERSTANDING
[PPR]  776,406 rows × 9 cols


,dtype
Date of Sale (dd/mm/yyyy),object
Address,object
County,object
Eircode,object
Price (),object
Not Full Market Price,object
VAT Exclusive,object
Description of Property,object
Property Size Description,object


,missing
Date of Sale (dd/mm/yyyy),0
Address,0
County,0
Eircode,549468
Price (),0
Not Full Market Price,0
VAT Exclusive,0
Description of Property,0
Property Size Description,723563


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN," 343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN," 185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN," 438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN," 400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN



[CPI] (16250, 5)  span: 1922 January → 2026 February
[Unemployment] (6084, 6)  metrics: ['Seasonally Adjusted Monthly Unemployment', 'Seasonally Adjusted Monthly Unemployment Rate']
[Earnings] (127008, 6)  quarters: 2008Q1 → 2025Q4
[Debt-ratio] (42, 5)
Data Understanding Completed


## Section 4: PPR Data Preprocessing

This section prepares the Property Price Register dataset for analysis by cleaning, standardizing, and transforming key variables.

### Column Standardization
Column names are cleaned by removing whitespace and then mapped to consistent, descriptive names using a rule-based approach. This ensures uniformity across different dataset versions.

### Data Type Conversion
- The `sale_date` column is converted to datetime format.
- The `price` column is cleaned to remove non-numeric characters and converted to numeric format.

Invalid or missing values in these critical fields are removed.

### Market Filtering
Transactions flagged as non-market sales are excluded to ensure that only valid open-market transactions are retained.

### Duplicate Removal
Duplicate records are identified and removed to prevent bias in the analysis.

### Feature Construction
Several binary indicators are created:
- `is_new`: Identifies newly built properties
- `is_apartment`: Identifies apartment units
- `vat_excl`: Indicates whether VAT is excluded

Additionally, property size categories are encoded into ordinal values to capture relative size differences.

### Final Validation
The cleaned dataset is summarised, including:
- Total number of valid records
- Date range of transactions
- Price range

This ensures that the dataset is consistent, reliable, and ready for integration with macroeconomic data.

In [29]:
print("="*65); print("SECTION 4 — PREPROCESSING PPR"); print("="*65)

ppr = ppr_raw.copy()
ppr.columns = [c.strip() for c in ppr.columns]
print("Raw columns:", ppr.columns.tolist())

# Robust rename (size BEFORE description to avoid collision)
col_map = {}
for c in ppr.columns:
    cl = c.lower()
    if   "date"    in cl:                     col_map[c]="sale_date"
    elif "address" in cl:                     col_map[c]="address"
    elif "county"  in cl:                     col_map[c]="county"
    elif "eircode" in cl:                     col_map[c]="eircode"
    elif "price"   in cl and "not" not in cl: col_map[c]="price"
    elif "not"     in cl and "market" in cl:  col_map[c]="not_full_market"
    elif "vat"     in cl:                     col_map[c]="vat_exclusive"
    elif "size"    in cl:                     col_map[c]="property_size"   # before description!
    elif "description" in cl:                 col_map[c]="property_type"

ppr.rename(columns=col_map, inplace=True)
ppr = ppr.loc[:, ~ppr.columns.duplicated()]
print("Renamed:", ppr.columns.tolist())

ppr["sale_date"] = pd.to_datetime(
    ppr.get("sale_date", pd.Series(dtype=str)), dayfirst=True, errors="coerce")
ppr["price"] = pd.to_numeric(
    ppr["price"].astype(str).str.replace(r"[^\d.]", "", regex=True), errors="coerce")
ppr.dropna(subset=["sale_date","price"], inplace=True)

if "not_full_market" in ppr.columns:
    mkt = ppr["not_full_market"].astype(str).str.strip().str.upper()
    print("not_full_market values:", mkt.unique())
    ppr = ppr[mkt.isin(["NO","N","FALSE","0"])]

before = len(ppr)
ppr.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(ppr):,}")
ppr = ppr[ppr["price"] > 0].copy()
ppr["county"] = ppr.get("county", pd.Series("Unknown", index=ppr.index)) \
                   .astype(str).str.strip().str.title()

pt = ppr.get("property_type", pd.Series("", index=ppr.index)).astype(str)
ppr["is_new"]       = pt.str.contains("New",       case=False, na=False).astype(int)
ppr["is_apartment"] = pt.str.contains("Apartment", case=False, na=False).astype(int)
ppr["vat_excl"] = (ppr.get("vat_exclusive", pd.Series("No", index=ppr.index))
                   .astype(str).str.strip().str.upper().isin(["YES","Y","TRUE","1"])).astype(int)

SIZE_MAP = {"less than 38 sq metres": 0,
            "greater than or equal to 38 sq metres and less than 125 sq metres": 1,
            "greater than or equal to 125 sq metres": 2}
ppr["size_cat"] = (ppr["property_size"].map(
    lambda x: next((v for k,v in SIZE_MAP.items() if isinstance(x,str) and k in x.lower()), np.nan))
    if "property_size" in ppr.columns else np.nan)

print(f"\n✔  PPR clean: {ppr.shape[0]:,} rows")
print(f"   Date  : {ppr['sale_date'].min().date()} → {ppr['sale_date'].max().date()}")
print(f"   Price : €{ppr['price'].min():,.0f} → €{ppr['price'].max():,.0f}")
display(ppr.head(3))

SECTION 4 — PREPROCESSING PPR
Raw columns: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (\x80)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']
Renamed: ['sale_date', 'address', 'county', 'eircode', 'price', 'not_full_market', 'vat_exclusive', 'property_type', 'property_size']
not_full_market values: ['NO' 'YES']
Duplicates removed: 812

✔  PPR clean: 736,002 rows
   Date  : 2010-01-01 → 2026-03-13
   Price : €5,031 → €387,665,198


,sale_date,address,county,eircode,price,not_full_market,vat_exclusive,property_type,property_size,is_new,is_apartment,vat_excl,size_cat
0,2010-01-01,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,343000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,0,1,0,NaN
1,2010-01-03,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,185000.0,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,1,1,1,1.0
2,2010-01-04,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,438500.0,No,No,Second-Hand Dwelling house /Apartment,NaN,0,1,0,NaN


## Section 5: Macroeconomic Data Preprocessing

This section prepares macroeconomic indicators for integration with the property dataset.

### Time Alignment
The date range of the PPR dataset is used as a reference window. All macroeconomic datasets are filtered to match this period to ensure temporal consistency.

### CPI Processing
The CPI dataset is:
- Parsed into datetime format
- Aggregated to monthly frequency
- Sorted chronologically

### Unemployment Processing
The unemployment dataset is filtered to retain:
- The unemployment rate metric
- Aggregate values across all genders

Monthly averages are computed to ensure consistency.

### Earnings Processing
The earnings dataset is:
- Parsed into quarterly timestamps
- Filtered to include all sectors and employees
- Aggregated to produce average weekly earnings

### Output
Each dataset is cleaned, aligned, and reduced to a consistent time index, making them suitable for merging with the PPR dataset.

In [30]:
print("="*65); print("SECTION 5 — MACROECONOMIC DATA PREPROCESSING"); print("="*65)

ppr_start = ppr["sale_date"].min().replace(day=1)
ppr_end   = ppr["sale_date"].max().replace(day=1)

cpi = cpi_raw.copy()
cpi["date"] = cpi["Month"].apply(parse_cso_month)
cpi.dropna(subset=["date","VALUE"], inplace=True)
cpi_monthly = (cpi.groupby("date")["VALUE"].mean().reset_index()
               .rename(columns={"VALUE":"cpi"}).sort_values("date"))
cpi_monthly = cpi_monthly[(cpi_monthly["date"]>=ppr_start) & (cpi_monthly["date"]<=ppr_end)].reset_index(drop=True)

unemp = unemp_raw.copy()
unemp["date"] = unemp["Month"].apply(parse_cso_month)
unemp.dropna(subset=["date","VALUE"], inplace=True)
unemp_rate = (unemp[
    unemp["Statistic Label"].str.contains("Unemployment Rate",case=False,na=False) &
    unemp["Sex"].str.strip().str.lower().isin(["both sexes","both","all"])]
    .groupby("date")["VALUE"].mean().reset_index()
    .rename(columns={"VALUE":"unemployment_rate"}).sort_values("date"))
unemp_rate = unemp_rate[(unemp_rate["date"]>=ppr_start) & (unemp_rate["date"]<=ppr_end)].reset_index(drop=True)

earn = earn_raw.copy()
earn["date"] = earn["Quarter"].apply(parse_quarter)
earn.dropna(subset=["date"], inplace=True)
awe = (earn[earn["Statistic Label"].str.contains("Average Weekly Earnings",case=False,na=False) &
            earn["Economic Sector NACE Rev 2"].str.strip().str.lower().str.contains("all nace",na=False) &
            earn["Type of Employee"].str.strip().str.lower().str.contains("all employees",na=False)]
       .groupby("date")["VALUE"].mean().reset_index()
       .rename(columns={"VALUE":"avg_weekly_earnings"}).sort_values("date"))
awe = awe[(awe["date"]>=ppr_start) & (awe["date"]<=ppr_end)].reset_index(drop=True)

print(f"CPI: {len(cpi_monthly)} | Unemployment: {len(unemp_rate)} | Earnings: {len(awe)}")
print("✔  Macro data ready.")

SECTION 5 — MACROECONOMIC DATA PREPROCESSING
CPI: 194 | Unemployment: 194 | Earnings: 64
✔  Macro data ready.


## Section 6: Data Integration

This section combines the cleaned PPR dataset with macroeconomic indicators to form a unified analytical dataset.

### Time Feature Creation
Two time-based keys are created:
- `year_month`: Monthly timestamp for CPI and unemployment merging
- `quarter_start`: Quarterly timestamp for earnings merging

### Dataset Merging
The PPR dataset is merged with:
- CPI data (monthly)
- Unemployment data (monthly)
- Earnings data (quarterly)

Left joins are used to preserve all property transactions.

### Missing Value Handling
Missing macroeconomic values are handled using:
- Forward-fill and backward-fill methods for continuity
- Default fallback values for any remaining gaps

### Final Dataset
The resulting dataset contains:
- Property transaction features
- Macroeconomic indicators aligned in time

This integrated dataset serves as the foundation for subsequent feature engineering and modelling.

In [31]:
print("="*65); print("SECTION 6 — DATA INTEGRATION"); print("="*65)

ppr["year_month"]    = ppr["sale_date"].dt.to_period("M").dt.to_timestamp()
ppr["quarter_start"] = ppr["sale_date"].dt.to_period("Q").dt.to_timestamp()

df = ppr.merge(cpi_monthly.rename(columns={"date":"year_month"}), on="year_month", how="left")
df = df.merge(unemp_rate.rename(columns={"date":"year_month"}),   on="year_month", how="left")
df = df.merge(awe.rename(columns={"date":"quarter_start"}),       on="quarter_start", how="left")
df.sort_values("sale_date", inplace=True)
df.reset_index(drop=True, inplace=True)
df[["cpi","unemployment_rate","avg_weekly_earnings"]] = (
    df[["cpi","unemployment_rate","avg_weekly_earnings"]].ffill().bfill())
for col,val in {"cpi":103.8,"unemployment_rate":7.5,"avg_weekly_earnings":800.0}.items():
    df[col] = df[col].fillna(val)

print(f"Merged: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

SECTION 6 — DATA INTEGRATION
Merged: 736,002 rows × 18 cols


,sale_date,address,county,eircode,price,not_full_market,vat_exclusive,property_type,property_size,is_new,is_apartment,vat_excl,size_cat,year_month,quarter_start,cpi,unemployment_rate,avg_weekly_earnings
0,2010-01-01,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,343000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,0,1,0,NaN,2010-01-01,2010-01-01,1326.576923,17.566667,662.518333
1,2010-01-03,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,185000.0,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,1,1,1,1.0,2010-01-01,2010-01-01,1326.576923,17.566667,662.518333
2,2010-01-04,"59 ormond keep, nenagh",Tipperary,NaN,128000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,0,1,0,NaN,2010-01-01,2010-01-01,1326.576923,17.566667,662.518333


In [32]:
### Checking for Null Values

df.isnull().sum()

sale_date                   0
address                     0
county                      0
eircode                520783
price                       0
not_full_market             0
vat_exclusive               0
property_type               0
property_size          684860
is_new                      0
is_apartment                0
vat_excl                    0
size_cat               691585
year_month                  0
quarter_start               0
cpi                         0
unemployment_rate           0
avg_weekly_earnings         0
dtype: int64

## Selected Feature Preview

The displayed subset of columns represents the key variables relevant to the objectives of this study. Specifically, the selected features include:

- Transaction details: `sale_date`, `price`
- Location identifier: `county`
- Property characteristics: `is_new`, `is_apartment`
- Macroeconomic indicators: `cpi`, `unemployment_rate`, `avg_weekly_earnings`

These variables were prioritised because they capture the core dimensions influencing housing prices — temporal trends, location effects, property type, and broader economic conditions.

Other variables such as `eircode`, `property_size`, and `size_cat` were excluded at this stage due to a high proportion of missing values (exceeding 70%), which could introduce noise and reduce model reliability if not handled appropriately.

Additionally, some columns were not included because they do not contribute meaningful information to the predictive task or fall outside the scope of this analysis.

This step provides a concise view of the most relevant features before proceeding to further feature engineering and modelling.

In [35]:
print(f"Merged: {df.shape[0]:,} rows × {df.shape[1]} cols")

display(df[[
    "sale_date",
    "county",
    "price",
    "is_new",
    "is_apartment",
    "cpi",
    "unemployment_rate",
    "avg_weekly_earnings"
]].head(3))

print("Selected Feature complete.")

Merged: 736,002 rows × 18 cols


,sale_date,county,price,is_new,is_apartment,cpi,unemployment_rate,avg_weekly_earnings
0,2010-01-01,Dublin,343000.0,0,1,1326.576923,17.566667,662.518333
1,2010-01-03,Laois,185000.0,1,1,1326.576923,17.566667,662.518333
2,2010-01-04,Tipperary,128000.0,0,1,1326.576923,17.566667,662.518333


Selected Feature complete.
